# Drainage maps for each dam

Makes one catchment and drainage map per dam from the catchments, streams and outlets in
`outputs/drainage.gpkg` and the NASADEM elevation model. Each map is laid out on an A1 landscape page and saved to
`outputs/maps/` as a 300 dpi JPEG and a PDF, with a 2000 px wide PNG preview in `figures/`.

## Load config and inputs

In [ ]:
from datetime import date
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
import yaml
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.ticker import FuncFormatter, MaxNLocator
from pyproj import CRS
from rasterio.features import geometry_mask
from rasterio.windows import from_bounds

ROOT = Path.cwd().parent
CONFIG_PATH = ROOT / "config" / "dams.yaml"
FIGURES_DIR = ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

dem_cfg = config["dem"]
dem_path = ROOT / "data" / f"dem_{dem_cfg['source'].lower()}_utm.tif"
gpkg_path = ROOT / "outputs" / "drainage.gpkg"
project_crs = CRS.from_user_input(config["crs"])

outlets = gpd.read_file(gpkg_path, layer="outlets").to_crs(project_crs)
catchments = gpd.read_file(gpkg_path, layer="catchments").to_crs(project_crs)
streams = gpd.read_file(gpkg_path, layer="streams").to_crs(project_crs)

dem_credit = f"DEM: {dem_cfg['source']} {dem_cfg['resolution_m']} m (OpenTopography)"
map_author = config.get("map_author")
map_date = date.today().isoformat()

## Coordinate system description

In [ ]:
# Build the coordinate system block from the project CRS
operation = project_crs.coordinate_operation
params = {p.name: p.value for p in operation.params}
crs_lines = [
    ("Coordinate system", project_crs.name),
    ("Projection", operation.method_name),
    ("Datum", project_crs.datum.name),
    ("False easting", f"{params['False easting']:,.0f}"),
    ("False northing", f"{params['False northing']:,.0f}"),
    ("Central meridian", f"{params['Longitude of natural origin']:g}"),
    ("Scale factor", f"{params['Scale factor at natural origin']:g}"),
    ("Latitude of origin", f"{params['Latitude of natural origin']:g}"),
    ("Units", project_crs.axis_info[0].unit_name),
]
for label, value in crs_lines:
    print(f"{label}: {value}")

## Map layout and styling

In [ ]:
# A1 landscape page size in inches, and the positions of the page elements in figure fractions
MM_PER_INCH = 25.4
FIG_SIZE = (841 / MM_PER_INCH, 594 / MM_PER_INCH)

# Ratio of the A1 page width to the A4 page width. Every text size, line width and marker size
# below is multiplied by it, so the page keeps the proportions of the A4 layout.
SCALE = 841 / 297

# Matplotlib defaults for tick marks, spines, label padding and marker edges, scaled the same way
plt.rcParams.update({
    "axes.linewidth": 0.8 * SCALE,
    "axes.labelpad": 4.0 * SCALE,
    "xtick.major.size": 3.5 * SCALE,
    "ytick.major.size": 3.5 * SCALE,
    "xtick.major.width": 0.8 * SCALE,
    "ytick.major.width": 0.8 * SCALE,
    "xtick.major.pad": 3.5 * SCALE,
    "ytick.major.pad": 3.5 * SCALE,
    "lines.markeredgewidth": 1.0 * SCALE,
    "patch.linewidth": 1.0 * SCALE,
})

# Output resolution and folders
DPI = 300
JPEG_QUALITY = 95
PREVIEW_WIDTH_PX = 2000
MAPS_DIR = ROOT / "outputs" / "maps"
MAPS_DIR.mkdir(parents=True, exist_ok=True)
MAP_AXES = [0.05, 0.27, 0.58, 0.62]
BUFFER_FRACTION = 0.10

CATCHMENT_COLOR = "black"
STREAM_COLOR = "tab:blue"
OUTLET_COLOR = "red"

# Elevation colours from dark green (low) through light green, pale yellow and orange to dark brown (high),
# with no blues so the blue streams stand out
ELEVATION_CMAP = LinearSegmentedColormap.from_list(
    "elevation", ["#1a7837", "#a6d96a", "#ffffbf", "#fdae61", "#a6611a", "#5c3317"]
)

# Map axes aspect ratio (width / height) in inches
MAP_ASPECT = (MAP_AXES[2] * FIG_SIZE[0]) / (MAP_AXES[3] * FIG_SIZE[1])

## Helper functions

In [ ]:
def map_extent(geometry, buffer_fraction, aspect):
    """Return a (west, south, east, north) extent around the geometry plus a buffer, widened to the given aspect."""
    west, south, east, north = geometry.bounds
    width, height = east - west, north - south
    west, east = west - buffer_fraction * width, east + buffer_fraction * width
    south, north = south - buffer_fraction * height, north + buffer_fraction * height

    # Widen the narrower dimension so the extent matches the map axes aspect ratio
    width, height = east - west, north - south
    if width / height < aspect:
        width = height * aspect
    else:
        height = width / aspect
    cx, cy = (west + east) / 2, (south + north) / 2
    return cx - width / 2, cy - height / 2, cx + width / 2, cy + height / 2


def read_dem_in_catchment(extent, geometry):
    """Read the DEM inside the extent, masked to the catchment polygon, and return it with its plotting extent."""
    with rasterio.open(dem_path) as src:
        window = from_bounds(*extent, transform=src.transform).round_offsets().round_lengths()
        elevation = src.read(1, window=window, masked=True, boundless=True)
        transform = src.window_transform(window)
        west, south, east, north = src.window_bounds(window)
    outside = geometry_mask([geometry], out_shape=elevation.shape, transform=transform)
    return np.ma.masked_where(outside | np.ma.getmaskarray(elevation), elevation), (west, east, south, north)


def scale_bar_length_km(map_width_m):
    """Pick a round scale bar length close to a quarter of the map width."""
    candidates_km = [0.1, 0.2, 0.25, 0.5, 1, 2, 2.5, 5, 10]
    target_km = map_width_m / 4000
    return min(candidates_km, key=lambda km: abs(km - target_km))


def add_scale_bar(fig, map_width_m, centre_x, bottom, height, segments=4):
    """Draw a scale bar of alternating black and white segments, in kilometres, centred at centre_x."""
    length_km = scale_bar_length_km(map_width_m)
    width_fraction = MAP_AXES[2] * (length_km * 1000) / map_width_m
    ax = fig.add_axes([centre_x - width_fraction / 2, bottom, width_fraction, height])
    segment_km = length_km / segments
    for k in range(segments):
        ax.add_patch(Rectangle((k * segment_km, 0), segment_km, 1,
                               facecolor="black" if k % 2 == 0 else "white",
                               edgecolor="black", linewidth=0.8 * SCALE))
    ax.set_xlim(0, length_km)
    ax.set_ylim(0, 1)
    # Ticks at every segment boundary, labelled at the start, middle and end only
    ax.set_xticks([k * segment_km for k in range(segments + 1)])
    ax.set_xticklabels(
        [f"{k * segment_km:g}" if k in (0, segments // 2, segments) else "" for k in range(segments + 1)],
        fontsize=8 * SCALE,
    )
    ax.set_yticks([])
    for side in ("top", "right", "left", "bottom"):
        ax.spines[side].set_visible(False)
    ax.tick_params(axis="x", length=3 * SCALE, width=0.8 * SCALE, pad=2 * SCALE)
    ax.set_xlabel("Kilometres", fontsize=8 * SCALE, labelpad=2 * SCALE)


def add_north_arrow(fig, ax, x, y_base, y_tip):
    """Draw a north arrow with the letter N above it, in figure coordinates."""
    ax.annotate(
        "", xy=(x, y_tip), xytext=(x, y_base), xycoords="figure fraction",
        arrowprops=dict(facecolor="black", edgecolor="black", width=6 * SCALE, headwidth=16 * SCALE),
        annotation_clip=False,
    )
    fig.text(x, y_tip + 0.015, "N", ha="center", va="bottom", fontsize=16 * SCALE, fontweight="bold")


def add_text_block(fig, lines, x_label, x_value, y_top, line_height, fontsize):
    """Write label/value pairs as two aligned columns starting at y_top."""
    for k, (label, value) in enumerate(lines):
        y = y_top - k * line_height
        fig.text(x_label, y, f"{label}:", ha="left", va="top", fontsize=fontsize)
        fig.text(x_value, y, value, ha="left", va="top", fontsize=fontsize)

## Make and save one map per dam

In [ ]:
for dam in config["dams"]:
    name = dam["name"]
    catchment = catchments[catchments["name"] == name].iloc[0]
    dam_streams = streams[streams["name"] == name]
    dam_outlet = outlets[outlets["name"] == name]

    extent = map_extent(catchment.geometry, BUFFER_FRACTION, MAP_ASPECT)
    elevation, image_extent = read_dem_in_catchment(extent, catchment.geometry)
    elev_min, elev_max = catchment["elev_min_m"], catchment["elev_max_m"]

    fig = plt.figure(figsize=FIG_SIZE, facecolor="white")

    # Neatline around the page
    fig.add_artist(Rectangle((0.01, 0.01), 0.98, 0.98, transform=fig.transFigure,
                             fill=False, edgecolor="black", linewidth=2 * SCALE))

    fig.text(0.5, 0.94, f"{name.upper()} CATCHMENT AND DRAINAGE MAP", ha="center",
             fontsize=18 * SCALE, fontweight="bold")

    # Map: DEM inside the catchment, catchment boundary, streams and outlet
    ax = fig.add_axes(MAP_AXES)
    ax.set_facecolor("white")
    ax.imshow(elevation, extent=image_extent, cmap=ELEVATION_CMAP, vmin=elev_min, vmax=elev_max,
              interpolation="nearest")
    gpd.GeoSeries([catchment.geometry], crs=project_crs).boundary.plot(
        ax=ax, color=CATCHMENT_COLOR, linewidth=2.5 * SCALE
    )
    dam_streams.plot(ax=ax, color=STREAM_COLOR, linewidth=1.2 * SCALE)
    ax.plot(dam_outlet.geometry.x, dam_outlet.geometry.y, marker="*", color=OUTLET_COLOR,
            markersize=18 * SCALE, markeredgecolor="black", linestyle="", zorder=4)
    ax.set_xlim(extent[0], extent[2])
    ax.set_ylim(extent[1], extent[3])

    # Light UTM grid lines with easting and northing labels in metres
    ax.xaxis.set_major_locator(MaxNLocator(5))
    ax.yaxis.set_major_locator(MaxNLocator(5))
    metres = FuncFormatter(lambda value, _: f"{value:,.0f}")
    ax.xaxis.set_major_formatter(metres)
    ax.yaxis.set_major_formatter(metres)
    ax.tick_params(labelsize=8 * SCALE)
    ax.tick_params(axis="y", labelrotation=90)
    for label in ax.get_yticklabels():
        label.set_verticalalignment("center")
    ax.grid(True, color="lightgrey", linestyle="-", linewidth=0.5 * SCALE)
    ax.set_axisbelow(False)
    ax.set_xlabel("Easting (m)", fontsize=9 * SCALE)
    ax.set_ylabel("Northing (m)", fontsize=9 * SCALE)

    # North arrow and coordinate system block, top right
    add_north_arrow(fig, ax, x=0.83, y_base=0.78, y_tip=0.86)
    add_text_block(fig, crs_lines, x_label=0.66, x_value=0.765, y_top=0.74,
                   line_height=0.028, fontsize=8 * SCALE)
    fig.text(0.66, 0.74 - len(crs_lines) * 0.028 - 0.01, dem_credit, ha="left", va="top",
             fontsize=7.5 * SCALE, style="italic")

    # Legend, bottom left: map symbols, then the elevation colour ramp
    legend_handles = [
        Line2D([], [], marker="*", color=OUTLET_COLOR, markeredgecolor="black", markersize=14 * SCALE,
               linestyle="", label="Catchment outlet"),
        Line2D([], [], color=STREAM_COLOR, linewidth=1.5 * SCALE, label="Drainage networks"),
        Line2D([], [], color=CATCHMENT_COLOR, linewidth=2.5 * SCALE, label="Watershed boundary"),
    ]
    fig.legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(0.04, 0.215),
               title="Legend", title_fontproperties={"weight": "bold", "size": 10 * SCALE},
               fontsize=9 * SCALE, frameon=False, alignment="left")
    fig.text(0.05, 0.105, "Elevation (m)", ha="left", va="center", fontsize=9 * SCALE)
    ramp_ax = fig.add_axes([0.055, 0.03, 0.018, 0.06])
    ramp_ax.imshow(np.linspace(1, 0, 100).reshape(-1, 1), cmap=ELEVATION_CMAP, aspect="auto")
    ramp_ax.set_axis_off()
    fig.text(0.08, 0.087, f"High: {elev_max:,.0f}", ha="left", va="center", fontsize=8.5 * SCALE)
    fig.text(0.08, 0.033, f"Low: {elev_min:,.0f}", ha="left", va="center", fontsize=8.5 * SCALE)

    # Statistics block, bottom centre above the scale bar
    stats_lines = [
        ("Catchment Area", f"{catchment['area_km2']:.3f} km²"),
        ("Longest Path", f"{catchment['longest_flow_path_km']:.2f} km"),
        ("Catchment Slope", f"{catchment['mean_slope_pct']:.1f} %"),
    ]
    if dam.get("runoff_factor") is not None:
        stats_lines.append(("Runoff factor Kr", f"{dam['runoff_factor']:g}"))
    stats_lines.append(("Stream threshold", f"{catchment['stream_threshold_km2']:g} km²"))
    add_text_block(fig, stats_lines, x_label=0.40, x_value=0.53, y_top=0.215,
                   line_height=0.025, fontsize=9.5 * SCALE)

    # Scale bar, bottom centre
    add_scale_bar(fig, map_width_m=extent[2] - extent[0], centre_x=0.50, bottom=0.06, height=0.012)

    # Author and date, bottom right
    if map_author:
        fig.text(0.96, 0.06, map_author, ha="right", va="bottom", fontsize=9 * SCALE)
    fig.text(0.96, 0.035, f"Date: {map_date}", ha="right", va="bottom", fontsize=9 * SCALE)

    # A1 JPEG and PDF in outputs/maps/, and a 2000 px wide PNG preview in figures/
    stem = f"{name.lower()}_drainage_map"
    jpg_path = MAPS_DIR / f"{stem}_A1.jpg"
    pdf_path = MAPS_DIR / f"{stem}_A1.pdf"
    preview_path = FIGURES_DIR / f"{stem}.png"
    fig.savefig(jpg_path, dpi=DPI, facecolor="white", pil_kwargs={"quality": JPEG_QUALITY})
    fig.savefig(pdf_path, dpi=DPI, facecolor="white")
    fig.savefig(preview_path, dpi=PREVIEW_WIDTH_PX / FIG_SIZE[0], facecolor="white")
    for path in (jpg_path, pdf_path, preview_path):
        print(f"Saved {path}")
    plt.show()